# Trabalho 1 — Aquisição de Dados (Cinema)

Coleta TMDB (API) + OMDb (API) + Letterboxd (scraping), integração e limpeza.

**Antes de rodar:** preencha `TMDB_API_KEY` e `OMDB_API_KEY` em `.env` (veja o README).

## 0. Setup e utilitários

In [ ]:
from __future__ import annotations

import json
import os
import time
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import requests
from bs4 import BeautifulSoup
from dotenv import load_dotenv
from tqdm.auto import tqdm

ROOT = Path.cwd()
if not (ROOT / "requirements.txt").exists() and (ROOT / "trabalho01" / "requirements.txt").exists():
    ROOT = ROOT / "trabalho01"

DIR_BRUTOS = ROOT / "dados_brutos"
DIR_TRATADOS = ROOT / "dados_tratados"
DIR_DOCS = ROOT / "docs"
LOG_PATH = DIR_DOCS / "proveniencia.jsonl"

for d in (DIR_BRUTOS, DIR_TRATADOS, DIR_DOCS):
    d.mkdir(parents=True, exist_ok=True)

load_dotenv(ROOT / ".env")
TMDB_API_KEY = os.getenv("TMDB_API_KEY", "").strip()
OMDB_API_KEY = os.getenv("OMDB_API_KEY", "").strip()

HEADERS = {
    "User-Agent": (
        "UFAM-CD-Trabalho1-Cinema/1.0 "
        "(+academic; contato via repositorio do projeto)"
    ),
    "Accept-Language": "en-US,en;q=0.9",
}

print("ROOT:", ROOT)
print("TMDB_API_KEY definida:", bool(TMDB_API_KEY))
print("OMDB_API_KEY definida:", bool(OMDB_API_KEY))
if not TMDB_API_KEY or not OMDB_API_KEY:
    print("AVISO: preencha trabalho01/.env com TMDB_API_KEY e OMDB_API_KEY antes da coleta.")

In [ ]:
def log_proveniencia(
    fonte: str,
    url: str,
    metodo: str,
    status: int | None,
    params: dict | None = None,
    observacao: str = "",
) -> None:
    """Append de uma linha JSON no registro de proveniência."""
    registro = {
        "timestamp": datetime.now(timezone.utc).isoformat(),
        "fonte": fonte,
        "url": url,
        "metodo": metodo,
        "params": params or {},
        "status": status,
        "observacao": observacao,
    }
    with LOG_PATH.open("a", encoding="utf-8") as f:
        f.write(json.dumps(registro, ensure_ascii=False) + "\n")


def get_com_retry(
    url: str,
    *,
    params: dict | None = None,
    headers: dict | None = None,
    timeout: int = 30,
    max_tentativas: int = 3,
    sleep_base: float = 1.0,
) -> requests.Response:
    """GET com retentativas para 429/5xx."""
    ultimo_erro: Exception | None = None
    for tentativa in range(1, max_tentativas + 1):
        try:
            resp = requests.get(
                url,
                params=params,
                headers=headers or HEADERS,
                timeout=timeout,
            )
            if resp.status_code in {429, 500, 502, 503, 504}:
                time.sleep(sleep_base * tentativa)
                continue
            return resp
        except requests.RequestException as exc:
            ultimo_erro = exc
            time.sleep(sleep_base * tentativa)
    if ultimo_erro:
        raise ultimo_erro
    raise RuntimeError(f"Falha ao obter {url}")


print("Utilitários carregados. Log:", LOG_PATH)

## 1. Teste das API keys (rode após preencher o `.env`)

In [ ]:
if TMDB_API_KEY:
    r_tmdb = get_com_retry(
        "https://api.themoviedb.org/3/movie/550",
        params={"api_key": TMDB_API_KEY},
    )
    print("TMDB:", r_tmdb.status_code, r_tmdb.json().get("title"))
    log_proveniencia(
        "TMDB",
        r_tmdb.url.replace(TMDB_API_KEY, "***"),
        "API GET /movie/550",
        r_tmdb.status_code,
        {"movie_id": 550},
        "teste de chave",
    )
else:
    print("TMDB: chave ausente")

if OMDB_API_KEY:
    r_omdb = get_com_retry(
        "https://www.omdbapi.com/",
        params={"i": "tt0137523", "apikey": OMDB_API_KEY},
    )
    data = r_omdb.json()
    print("OMDb:", r_omdb.status_code, data.get("Title"), data.get("imdbRating"), data.get("Metascore"))
    log_proveniencia(
        "OMDb",
        "https://www.omdbapi.com/?i=tt0137523&apikey=***",
        "API GET",
        r_omdb.status_code,
        {"i": "tt0137523"},
        "teste de chave",
    )
else:
    print("OMDb: chave ausente")

## 2. Coleta TMDB (API)

- `discover/movie` com `sort_by=revenue.desc`, páginas 1–50 (~1000 IDs)
- Detalhes em `/movie/{id}` para cada filme
- Salva **todos** os registros em `dados_brutos/tmdb_raw.csv` (sem filtrar `imdb_id`; isso fica para a OMDb)
- Proveniência em `docs/proveniencia.jsonl` (API key mascarada)
- Esta célula **substitui** o CSV bruto completo (expansão 400→1000)

In [ ]:
assert TMDB_API_KEY, "TMDB_API_KEY ausente no .env"

TMDB_DISCOVER_URL = "https://api.themoviedb.org/3/discover/movie"
TMDB_DETAIL_URL = "https://api.themoviedb.org/3/movie/{movie_id}"
TMDB_PAGES = range(1, 51)  # 50 páginas × 20 resultados = ~1000
TMDB_SLEEP = 0.25
TMDB_RAW_PATH = DIR_BRUTOS / "tmdb_raw.csv"

# --- Discover: IDs únicos ---
tmdb_ids: list[int] = []
seen: set[int] = set()

for page in tqdm(list(TMDB_PAGES), desc="TMDB discover"):
    params = {
        "api_key": TMDB_API_KEY,
        "sort_by": "revenue.desc",
        "page": page,
        "include_adult": "false",
    }
    resp = get_com_retry(TMDB_DISCOVER_URL, params=params)
    log_proveniencia(
        "TMDB",
        f"{TMDB_DISCOVER_URL}?sort_by=revenue.desc&page={page}&api_key=***",
        "API GET /discover/movie",
        resp.status_code,
        {"sort_by": "revenue.desc", "page": page, "include_adult": False},
    )
    resp.raise_for_status()
    for item in resp.json().get("results", []):
        mid = item.get("id")
        if mid is not None and mid not in seen:
            seen.add(mid)
            tmdb_ids.append(mid)
    time.sleep(TMDB_SLEEP)

print(f"IDs únicos coletados no discover: {len(tmdb_ids)}")

# --- Details: todos os filmes (com ou sem imdb_id) ---
rows: list[dict] = []

for mid in tqdm(tmdb_ids, desc="TMDB details"):
    url = TMDB_DETAIL_URL.format(movie_id=mid)
    resp = get_com_retry(url, params={"api_key": TMDB_API_KEY})
    log_proveniencia(
        "TMDB",
        f"{url}?api_key=***",
        "API GET /movie/{id}",
        resp.status_code,
        {"movie_id": mid},
    )
    if resp.status_code != 200:
        time.sleep(TMDB_SLEEP)
        continue
    data = resp.json()
    genres = "|".join(g.get("name", "") for g in data.get("genres") or [])
    rows.append(
        {
            "tmdb_id": data.get("id"),
            "imdb_id": data.get("imdb_id") or "",
            "title": data.get("title"),
            "release_date": data.get("release_date"),
            "budget": data.get("budget"),
            "revenue": data.get("revenue"),
            "genres": genres,
            "runtime": data.get("runtime"),
            "original_language": data.get("original_language"),
            "tmdb_vote_average": data.get("vote_average"),
            "tmdb_vote_count": data.get("vote_count"),
        }
    )
    time.sleep(TMDB_SLEEP)

df_tmdb = pd.DataFrame(rows)
df_tmdb.to_csv(TMDB_RAW_PATH, index=False, encoding="utf-8")
print(f"Salvo: {TMDB_RAW_PATH}")
print(f"Linhas: {len(df_tmdb)} | Colunas: {list(df_tmdb.columns)}")
print(f"Com imdb_id: {(df_tmdb['imdb_id'].astype(str).str.len() > 0).sum()}")
print(f"Sem imdb_id: {(df_tmdb['imdb_id'].astype(str).str.len() == 0).sum()}")
df_tmdb.head()


## 3. Coleta OMDb (API) — notas IMDb / Metascore

- Lê `tmdb_raw.csv` e usa **apenas** linhas com `imdb_id`
- `GET https://www.omdbapi.com/?i={imdb_id}&apikey=***`
- Salva `dados_brutos/omdb_raw.csv` (bruto; progresso incremental a cada 50 filmes)
- Cota free: 1000 req/dia — ~991 ids cabem em uma passada se a cota estiver livre


In [ ]:
assert OMDB_API_KEY, "OMDB_API_KEY ausente no .env"

OMDB_URL = "https://www.omdbapi.com/"
OMDB_SLEEP = 0.35
OMDB_RAW_PATH = DIR_BRUTOS / "omdb_raw.csv"
CHECKPOINT_EVERY = 50

df_tmdb_src = pd.read_csv(DIR_BRUTOS / "tmdb_raw.csv", dtype={"imdb_id": str})
imdb_ids = (
    df_tmdb_src["imdb_id"]
    .fillna("")
    .astype(str)
    .str.strip()
)
imdb_ids = [i for i in imdb_ids.unique().tolist() if i and i.lower() != "nan"]
print(f"imdb_id únicos para OMDb: {len(imdb_ids)}")

# Retoma se já existir parcial
done: set[str] = set()
rows_omdb: list[dict] = []
if OMDB_RAW_PATH.exists():
    prev = pd.read_csv(OMDB_RAW_PATH, dtype={"imdb_id": str})
    rows_omdb = prev.to_dict(orient="records")
    done = set(prev["imdb_id"].astype(str).tolist())
    print(f"Retomando: {len(done)} já coletados")

pendentes = [i for i in imdb_ids if i not in done]
print(f"Pendentes: {len(pendentes)}")

for n, iid in enumerate(tqdm(pendentes, desc="OMDb"), start=1):
    resp = get_com_retry(OMDB_URL, params={"i": iid, "apikey": OMDB_API_KEY})
    log_proveniencia(
        "OMDb",
        f"{OMDB_URL}?i={iid}&apikey=***",
        "API GET",
        resp.status_code,
        {"i": iid},
    )
    data = resp.json() if resp.status_code == 200 else {}
    # Cota esgotada / erro de autenticação
    if data.get("Error") and "limit" in str(data.get("Error")).lower():
        print("Cota OMDb atingida. Salvando progresso e parando.")
        break
    rows_omdb.append(
        {
            "imdb_id": iid,
            "imdb_rating": data.get("imdbRating"),
            "imdb_votes": data.get("imdbVotes"),
            "metascore": data.get("Metascore"),
            "omdb_response": data.get("Response"),
            "omdb_error": data.get("Error"),
            "url_coletada": f"{OMDB_URL}?i={iid}&apikey=***",
        }
    )
    if n % CHECKPOINT_EVERY == 0:
        pd.DataFrame(rows_omdb).to_csv(OMDB_RAW_PATH, index=False, encoding="utf-8")
    time.sleep(OMDB_SLEEP)

df_omdb = pd.DataFrame(rows_omdb)
df_omdb.to_csv(OMDB_RAW_PATH, index=False, encoding="utf-8")
print(f"Salvo: {OMDB_RAW_PATH}")
print(f"Linhas: {len(df_omdb)}")
ok = (df_omdb.get("omdb_response") == "True").sum() if "omdb_response" in df_omdb.columns else "n/a"
print(f"Response=True: {ok}")
df_omdb.head()


## 4. Scraping Letterboxd (HTML)

- URL: `https://letterboxd.com/tmdb/{tmdb_id}/` (robots.txt permite)
- Extrai `aggregateRating` do JSON-LD (`ratingValue`, `ratingCount`)
- `sleep(2)` entre requisições; checkpoint a cada 50 filmes
- Salva `dados_brutos/letterboxd_raw.csv`


In [ ]:
import re

LB_SLEEP = 2.0
LB_RAW_PATH = DIR_BRUTOS / "letterboxd_raw.csv"
CHECKPOINT_EVERY = 50


def parse_letterboxd_html(html: str) -> dict:
    soup = BeautifulSoup(html, "lxml")
    out = {
        "letterboxd_rating": None,
        "letterboxd_rating_count": None,
        "letterboxd_title": None,
    }
    script = soup.select_one('script[type="application/ld+json"]')
    if script:
        raw = script.string or script.get_text() or ""
        m = re.search(r"\{.*\}", raw, re.S)
        if m:
            try:
                data = json.loads(m.group(0))
                out["letterboxd_title"] = data.get("name")
                agg = data.get("aggregateRating") or {}
                out["letterboxd_rating"] = agg.get("ratingValue")
                out["letterboxd_rating_count"] = agg.get("ratingCount")
            except json.JSONDecodeError:
                pass
    if out["letterboxd_rating"] is None:
        tw = soup.select_one('meta[name="twitter:data2"]')
        if tw and tw.get("content"):
            mm = re.search(r"([0-9]+(?:\.[0-9]+)?)", tw["content"])
            if mm:
                out["letterboxd_rating"] = float(mm.group(1))
    return out


df_tmdb_lb = pd.read_csv(DIR_BRUTOS / "tmdb_raw.csv")
tmdb_ids_lb = df_tmdb_lb["tmdb_id"].dropna().astype(int).unique().tolist()
print(f"tmdb_id para Letterboxd: {len(tmdb_ids_lb)}")

done_lb: set[int] = set()
rows_lb: list[dict] = []
if LB_RAW_PATH.exists():
    prev_lb = pd.read_csv(LB_RAW_PATH)
    rows_lb = prev_lb.to_dict(orient="records")
    done_lb = set(prev_lb["tmdb_id"].dropna().astype(int).tolist())
    print(f"Retomando: {len(done_lb)} já coletados")

pendentes_lb = [i for i in tmdb_ids_lb if i not in done_lb]
print(f"Pendentes: {len(pendentes_lb)}")

for n, tid in enumerate(tqdm(pendentes_lb, desc="Letterboxd"), start=1):
    url = f"https://letterboxd.com/tmdb/{tid}/"
    try:
        resp = get_com_retry(url, timeout=40)
        status = resp.status_code
        final_url = str(resp.url)
        parsed = parse_letterboxd_html(resp.text) if status == 200 else {}
    except Exception as exc:
        status = None
        final_url = url
        parsed = {}
        log_proveniencia("Letterboxd", url, "SCRAPE GET", None, {"tmdb_id": tid}, str(exc)[:200])
        rows_lb.append(
            {
                "tmdb_id": tid,
                "letterboxd_rating": None,
                "letterboxd_rating_count": None,
                "letterboxd_title": None,
                "url_final": final_url,
                "http_status": status,
            }
        )
        time.sleep(LB_SLEEP)
        continue

    log_proveniencia(
        "Letterboxd",
        url,
        "SCRAPE GET",
        status,
        {"tmdb_id": tid, "final_url": final_url},
    )
    rows_lb.append(
        {
            "tmdb_id": tid,
            "letterboxd_rating": parsed.get("letterboxd_rating"),
            "letterboxd_rating_count": parsed.get("letterboxd_rating_count"),
            "letterboxd_title": parsed.get("letterboxd_title"),
            "url_final": final_url,
            "http_status": status,
        }
    )
    if n % CHECKPOINT_EVERY == 0:
        pd.DataFrame(rows_lb).to_csv(LB_RAW_PATH, index=False, encoding="utf-8")
    time.sleep(LB_SLEEP)

df_lb = pd.DataFrame(rows_lb)
df_lb.to_csv(LB_RAW_PATH, index=False, encoding="utf-8")
print(f"Salvo: {LB_RAW_PATH}")
print(f"Linhas: {len(df_lb)}")
print(f"Com rating: {df_lb['letterboxd_rating'].notna().sum()}")
df_lb.head()


## 5. Integração e limpeza

Carrega os três CSVs brutos **sem alterá-los**, faz left joins e limpa só a cópia tratada.

1. `tmdb` ⟕ `omdb` em `imdb_id`
2. resultado ⟕ `letterboxd` em `tmdb_id`
3. tipos, `budget`/`revenue` 0 → ausente, dedupe por `tmdb_id`
4. salva `dados_tratados/base_tratada.parquet` (+ CSV de apoio)


In [ ]:
# Carregar brutos (somente leitura)
df_tmdb = pd.read_csv(DIR_BRUTOS / "tmdb_raw.csv")
df_omdb = pd.read_csv(DIR_BRUTOS / "omdb_raw.csv")
df_lb = pd.read_csv(DIR_BRUTOS / "letterboxd_raw.csv")

print("Linhas brutas:")
print(f"  TMDB:       {len(df_tmdb)}")
print(f"  OMDb:       {len(df_omdb)}")
print(f"  Letterboxd: {len(df_lb)}")

# Cobertura das chaves antes do join
tmdb_com_imdb = df_tmdb["imdb_id"].notna() & (df_tmdb["imdb_id"].astype(str).str.strip() != "")
print(f"\nTMDB com imdb_id: {tmdb_com_imdb.sum()} / {len(df_tmdb)}")
print(f"TMDB sem imdb_id (não casam com OMDb): {(~tmdb_com_imdb).sum()}")

omdb_ids = set(df_omdb["imdb_id"].dropna().astype(str))
tmdb_ids_imdb = set(df_tmdb.loc[tmdb_com_imdb, "imdb_id"].astype(str))
casam_omdb = tmdb_ids_imdb & omdb_ids
print(f"imdb_id TMDB presentes no OMDb: {len(casam_omdb)}")
print(f"imdb_id TMDB ausentes no OMDb:  {len(tmdb_ids_imdb - omdb_ids)}")

lb_com_rating = df_lb["letterboxd_rating"].notna().sum()
print(f"\nLetterboxd com rating: {lb_com_rating} / {len(df_lb)}")
print(f"Letterboxd http_status != 200: {(df_lb['http_status'] != 200).sum()}")


In [ ]:
# Left joins
omdb_cols = [
    "imdb_id",
    "imdb_rating",
    "imdb_votes",
    "metascore",
    "omdb_response",
    "omdb_error",
]
lb_cols = [
    "tmdb_id",
    "letterboxd_rating",
    "letterboxd_rating_count",
    "letterboxd_title",
    "url_final",
    "http_status",
]

df = df_tmdb.merge(df_omdb[omdb_cols], on="imdb_id", how="left", indicator="merge_omdb")
df = df.merge(df_lb[lb_cols], on="tmdb_id", how="left", indicator="merge_lb")

print("Cobertura após joins (left):")
print(df["merge_omdb"].value_counts(dropna=False).to_string())
print(df["merge_lb"].value_counts(dropna=False).to_string())
print(f"\nCom imdb_rating:         {df['imdb_rating'].notna().sum()}")
print(f"Com metascore:          {df['metascore'].notna().sum()}")
print(f"Com letterboxd_rating:  {df['letterboxd_rating'].notna().sum()}")

# Limpeza (só na cópia tratada)
df["release_date"] = pd.to_datetime(df["release_date"], errors="coerce")

for col in ("budget", "revenue", "runtime", "tmdb_vote_count", "tmdb_id"):
    df[col] = pd.to_numeric(df[col], errors="coerce")

df["tmdb_vote_average"] = pd.to_numeric(df["tmdb_vote_average"], errors="coerce")
df["imdb_rating"] = pd.to_numeric(df["imdb_rating"], errors="coerce")
df["metascore"] = pd.to_numeric(df["metascore"], errors="coerce")
df["letterboxd_rating"] = pd.to_numeric(df["letterboxd_rating"], errors="coerce")
df["letterboxd_rating_count"] = pd.to_numeric(df["letterboxd_rating_count"], errors="coerce")

# imdb_votes vem como "1,486,308"
df["imdb_votes"] = (
    df["imdb_votes"]
    .astype(str)
    .str.replace(",", "", regex=False)
    .replace({"nan": pd.NA, "None": pd.NA, "": pd.NA})
)
df["imdb_votes"] = pd.to_numeric(df["imdb_votes"], errors="coerce")

# budget/revenue 0 = ausente na TMDB
n_budget0 = (df["budget"] == 0).sum()
n_revenue0 = (df["revenue"] == 0).sum()
df.loc[df["budget"] == 0, "budget"] = pd.NA
df.loc[df["revenue"] == 0, "revenue"] = pd.NA
print(f"\nbudget==0 → NA: {n_budget0}")
print(f"revenue==0 → NA: {n_revenue0}")

# Deduplicar por tmdb_id
antes = len(df)
df = df.drop_duplicates(subset=["tmdb_id"], keep="first")
print(f"Duplicatas tmdb_id removidas: {antes - len(df)}")

# Colunas auxiliares de merge só para diagnóstico; manter flags úteis
df["matched_omdb"] = df["merge_omdb"] == "both"
df["matched_letterboxd"] = df["merge_lb"] == "both"
df = df.drop(columns=["merge_omdb", "merge_lb"])

# Ordem de colunas
cols_final = [
    "tmdb_id",
    "imdb_id",
    "title",
    "release_date",
    "budget",
    "revenue",
    "genres",
    "runtime",
    "original_language",
    "tmdb_vote_average",
    "tmdb_vote_count",
    "imdb_rating",
    "imdb_votes",
    "metascore",
    "letterboxd_rating",
    "letterboxd_rating_count",
    "letterboxd_title",
    "matched_omdb",
    "matched_letterboxd",
    "omdb_response",
    "omdb_error",
    "url_final",
    "http_status",
]
df = df[[c for c in cols_final if c in df.columns]]

out_parquet = DIR_TRATADOS / "base_tratada.parquet"
out_csv = DIR_TRATADOS / "base_tratada.csv"
df.to_parquet(out_parquet, index=False)
df.to_csv(out_csv, index=False, encoding="utf-8")

log_proveniencia(
    "integracao",
    str(out_parquet),
    "JOIN_CLEAN",
    200,
    {
        "n_tmdb": int(len(df_tmdb)),
        "n_omdb": int(len(df_omdb)),
        "n_letterboxd": int(len(df_lb)),
        "n_final": int(len(df)),
        "matched_omdb": int(df["matched_omdb"].sum()),
        "matched_letterboxd": int(df["matched_letterboxd"].sum()),
        "budget0_to_na": int(n_budget0),
        "revenue0_to_na": int(n_revenue0),
    },
    "TMDB left OMDb(imdb_id) left Letterboxd(tmdb_id); budget/revenue 0→NA; dedupe tmdb_id",
)

print(f"\nSalvo: {out_parquet}")
print(f"Salvo: {out_csv}")
print(f"Linhas finais: {len(df)}")
print(f"Colunas: {list(df.columns)}")
df.head()
